### 1. Preliminaries

In [1]:
import glob
import wandb

try: import fastkaggle
except ModuleNotFoundError:
    !pip install -Uq fastkaggle

from fastkaggle import *
from fastai.vision.all import *
set_seed(42)

#competition = 'Paddy Doctor: Paddy Disease Classification'
#path = setup_comp(competition, install='fastai "timm>=0.6.2.dev0"')
path = Path('/kaggle/input/paddy-doctor-disease/Augmented and split - 26000 augmented images split into train (80) sets')

# competition = 'paddy-doctor-diseases-small-400-split'
# path = setup_comp(competition, install='fastai "timm>=0.6.2.dev0"')
# print(path)

# train images
train_path = path / 'train'
train_files = get_image_files(train_path)

# test images
test_path = path/'test'
test_files = get_image_files(test_path).sorted()

# train labels
train_df = pd.read_csv(path / 'metadata-train.csv')
print(train_df.shape)
train_df.label.value_counts()

(20800, 4)


label
brown_spot                  1601
downy_mildew                1601
white_stem_borer            1601
blast                       1600
bacterial_leaf_streak       1600
leaf_roller                 1600
tungro                      1600
black_stem_borer            1600
hispa                       1600
yellow_stem_borer           1600
bacterial_leaf_blight       1599
bacterial_panicle_blight    1599
normal                      1599
Name: count, dtype: int64

### 2. Dataloaders

In [2]:
dblock = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    get_y=parent_label,
    splitter=RandomSplitter(0.2, seed=42),
    item_tfms=Resize(480, method='squish'),
    batch_tfms=aug_transforms(size=224, min_scale=0.75)
)

dls = dblock.dataloaders(train_path)

In [3]:
dls = ImageDataLoaders.from_folder(
    train_path, 
    valid_pct=0.2,
    seed=42,
    item_tfms=Resize(480, method='squish'),
    batch_tfms=aug_transforms(size=224, min_scale=0.75)
)

In [4]:
wandb.finish()

In [5]:


wandb.login(key="f54d876ebca391d5a1ed3eebb7a43aa7bea9da21")

run = wandb.init(
    project="project-ablations",  # Customize as needed
    name="resnet34",
    config={
        "epochs": 100,
        "base_lr": 0.005,
        "weight_decay": 0.01,
        "architecture": "ResNet35"
    },
    reinit=True
)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: nigibril1 (nigibril1-gibril-tech). Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


### 3. Model training

In [6]:
learn = vision_learner(dls, resnet34, metrics=error_rate).to_fp16()

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth
100%|██████████| 83.3M/83.3M [00:00<00:00, 187MB/s] 


And that's it, 16 epochs to get the best baseline for the price 

In [ ]:
from timm import create_model
from fastai.vision.all import vision_learner, error_rate
from fastai.callback.wandb import WandbCallback
import os
os.makedirs("/kaggle/working/models", exist_ok=True)

# Add F1Score to metrics
learn = vision_learner(dls, 'resnet34', metrics=[error_rate, accuracy, 
                              Precision(average='macro'), 
                              Recall(average='macro'), 
                              F1Score(average='macro')],
                      model_dir=Path("/kaggle/working/models"))

learn.fine_tune(100, 0.005,cbs=[
        SaveModelCallback(monitor='f1_score', comp=np.greater, fname='resnet34_best_f1'),
        ShowGraphCallback(),
        EarlyStoppingCallback(monitor='f1_score', comp=np.greater, patience=10),
        WandbCallback(log_model=True)])
#learn.fine_tune(2, 0.005)



model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

epoch,train_loss,valid_loss,error_rate,accuracy,precision_score,recall_score,f1_score,time


In [ ]:
wandb.save("resnet34_best_f1")

### 4. Predictions

In [ ]:
# Get predictions on validation set
probs, target = learn.get_preds(dl=dls.valid)
error_rate(probs, target)


In [ ]:
# Get TTA predictions on validation set
probs, target = learn.tta(dl=dls.valid)
error_rate(probs, target)

#### Predictions on test set

In [ ]:
# test images
test_path = path/'test'
test_files = get_image_files(test_path).sorted()
test_classes = [f.parent.name for f in test_files]

probs, _ = learn.tta(dl=dls.test_dl(test_files))
preds = probs.argmax(dim=1)
pred_classes = dls.vocab[preds]

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

cls_report = classification_report(test_classes, pred_classes, 
                                   digits=5)
print(cls_report)
acc = accuracy_score(test_classes, pred_classes)

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

def plot_heatmap(y_true, y_pred, class_names, ax, title):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(
        cm, 
        annot=True, 
        square=True, 
        xticklabels=class_names, 
        yticklabels=class_names,
        fmt='d', 
        cmap=plt.cm.Blues,
        cbar=False,
        ax=ax
    )
    #ax.set_title(title, fontsize=16)
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=12, rotation=45, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=12)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_xlabel('Predicted Label', fontsize=12)

fig, ax = plt.subplots(1, 1, figsize=(8, 6))

class_names = dls.vocab
plot_heatmap(test_classes, pred_classes, class_names, ax, title="Resnet34")    

#fig.suptitle("Confusion Matrix Model Comparison", fontsize=12)
#fig.tight_layout()
#fig.subplots_adjust(top=1.25)
plt.show()
cm = confusion_matrix(test_classes, pred_classes)
print(cm)

In [ ]:
temp = pd.DataFrame({"y_true":test_classes,
                      "y_pred":pred_classes})
temp.to_csv('result.csv', index=False)
temp

#### Acknowledgements
1. https://www.kaggle.com/code/fmussari/fast-resnet34-with-fastai
